In [7]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os



In [10]:
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated  

In [11]:
class State(TypedDict):
    messages: Annotated[list, add_messages]   

In [2]:
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool when you need to perform calculations.
    
    Args:
        expression: A math expression like '2 + 2' or '100 * 0.15'
    """
    try:
        result = eval(expression)           
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

In [3]:
@tool
def search_web(query: str) -> str:
    """
    Search the web for current information.
    Use when asked about recent events, news, or facts you're unsure about.
    
    Args:
        query: The search query (e.g., 'latest AI news 2025')
    """
    # Mock implementation — replace with real search in production
    results = {
        "weather today": "Sunny, 25°C in most areas",
        "latest ai news": "OpenAI released GPT-5, Google launched Gemini 3.0",
        "stock market": "S&P 500 up 1.2% today, tech sector leading"
    }
    for key, value in results.items():
        if key in query.lower():
            return value
    return f"Search results for '{query}': No specific results found. This is a mock search."

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    Use when asked about weather conditions.
    
    Args:
        city: The city name (e.g., 'London', 'Tokyo', 'New York')
    """
    weather_data = {
        "london": "Cloudy, 15°C, 60% humidity",
        "tokyo": "Sunny, 28°C, 45% humidity",
        "new york": "Rainy, 18°C, 80% humidity",
        "paris": "Partly cloudy, 20°C, 55% humidity",
    }
    return weather_data.get(city.lower(), f"Weather data unavailable for {city}")

In [5]:
tools = [ search_web, get_weather]

In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [9]:
llm_with_tools = llm.bind_tools(tools)

In [12]:
def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm_with_tools.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}